# Kalkulator mocy produkcyjnych

Notebook porównuje **zapotrzebowanie na pracowników** na stacjach linii montażowej
z **liczbą obecnych pracowników** na bieżącej zmianie. Wynik zapisywany jest do pliku
Excel, z którego korzysta raport Power BI.

Kroki:
1. **ZPP_PLAN** – kolejność pojazdów na linii → matryca: który pojazd stoi na której stacji w kolejnych taktach.
2. **ZPP_SOP** – numer zlecenia ostatniego pojazdu w każdej pozycji kontraktu.
3. **OPERACJE_ZP03** – czasy montażu → liczba potrzebnych osób na stacji.
4. **ZASOBY** – obecni pracownicy bieżącej brygady → bilans mocy.

Ścieżki do plików są w prywatnym pliku `.config.ini` (wzór: `config.example.ini`).

In [ ]:
import datetime

import pandas as pd

from funkcje import (wczytaj_sciezki, ustal_obecna_zmiane, tekst_na_liczbe,
                     uwzglednij_wypozyczenia, zapisz_arkusz, zapisz_do_bazy)

pd.set_option('display.max_colwidth', None)

# ścieżki do plików wejściowych i pliku wynikowego
sciezki = wczytaj_sciezki()

# czas taktu linii [min] - czas montażu dzielony przez takt daje liczbę potrzebnych osób
takt = 95

## 1. ZPP_PLAN – plan produkcji

In [ ]:
df_zpp_plan_mc01 = pd.read_csv(sciezki['zpp_plan'], encoding='utf-8', sep=';', header=0)

# zostawiamy tylko stację MC01 - od niej liczona jest kolejka na całej linii
# .copy() - pracujemy na osobnej kopii, a nie na fragmencie poprzedniej tabeli
df_zpp_plan_mc01 = df_zpp_plan_mc01[df_zpp_plan_mc01['Stanow. robocze'] == 'MC01'].copy()

# SAP zapisuje pustą datę jako '00.00.0000' - zamieniamy ją na brak wartości
df_zpp_plan_mc01['Rzeczywista data zakończenia'] = df_zpp_plan_mc01['Rzeczywista data zakończenia'].replace('00.00.0000', pd.NA)
df_zpp_plan_mc01['Rzeczywista data zakończenia'] = pd.to_datetime(df_zpp_plan_mc01['Rzeczywista data zakończenia'], format='%d.%m.%Y')
df_zpp_plan_mc01['Planowana data wejścia'] = pd.to_datetime(df_zpp_plan_mc01['Planowana data wejścia'], format='%d.%m.%Y')

# kolejność pojazdów wg planowanej daty i godziny wejścia na linię
df_zpp_plan_mc01 = df_zpp_plan_mc01.sort_values(by=['Planowana data wejścia', 'Planowana godzina wejścia'])

# usunięcie zakończonych zleceń poza ostatnimi 20
# (pojazdy, które opuściły MC01, są jeszcze na dalszych stacjach linii)
# max(..., 0) - gdy zakończonych jest mniej niż 20, nie usuwamy nic
trim = max(df_zpp_plan_mc01['Rzeczywista data zakończenia'].count() - 20, 0)
df_zpp_plan_mc01_trimmed = df_zpp_plan_mc01.iloc[trim:].reset_index(drop=True)

# ograniczenie wyników wyłącznie do 100 kolejnych pojazdów
df_zpp_plan_mc01_trimmed = df_zpp_plan_mc01_trimmed.iloc[:100]

### Kolejka pojazdów na stacjach

Każda stacja ma swoją pozycję w kolejce linii. Pojazd z wiersza `i` planu ma na stacji
pozycję `pozycja_stacji + i`. Stacje z tą samą pozycją pracują równolegle.

In [ ]:
# pozycja każdej stacji w kolejce linii montażowej
kolejka = {'PC01.165': 1, 'MC01.165': 3, 'MC02.170': 4, 'MC03.180': 5, 'MC04.185': 6, 'PC05.235': 8,
           'MC05.190': 7, 'MC06.200': 8, 'MC07.205': 9, 'MC08.206': 10, 'PC09.210': 11, 'MC09.210': 11,
           'PC10.211': 12, 'MC10.215': 12, 'MC11.220': 13, 'MC12.240': 14, 'MC13.250': 15, 'MC14.270': 16,
           'MC15.275': 17, 'MC16.285': 18, 'MC17.287': 19}

# dla każdej stacji kolumna z kolejnymi pozycjami: pozycja, pozycja+1, pozycja+2, ...
liczba_pojazdow = len(df_zpp_plan_mc01_trimmed)
dict_kolejka = {}
for stacja, pozycja in kolejka.items():
    dict_kolejka[stacja] = list(range(pozycja, pozycja + liczba_pojazdow))

df_kolejka = pd.DataFrame(dict_kolejka)

# połączenie poziome planu z kolejką (wiersz po wierszu)
df_zpp_plan_mc01_kolejka = pd.concat([df_zpp_plan_mc01_trimmed, df_kolejka], axis=1)

### Matryca: takt × stacja

Dla każdej stacji i każdego taktu szukamy pojazdu, który w tym takcie stoi na stacji.
Jeśli na stacji nie ma pojazdu z planu (początek lub koniec horyzontu), wpisujemy zera.

In [ ]:
# pozycja w kolejce, od której zaczynamy (takt 0)
# wiersz 21 na MC01.165 minus 2 = w takcie 0 na MC01.165 stoi ostatni z 20 zakończonych pojazdów
poczatkowa_kolejka = df_zpp_plan_mc01_kolejka.loc[21, 'MC01.165'] - 2

wiersze_matrycy = []
for stacja in kolejka.keys():
    pozycja = poczatkowa_kolejka
    for takt_nr in range(liczba_pojazdow):
        # pojazd, który w tym takcie ma na stacji szukaną pozycję
        pojazd = df_zpp_plan_mc01_kolejka[df_zpp_plan_mc01_kolejka[stacja] == pozycja]

        if pojazd.empty:
            wiersze_matrycy.append({
                'Takt': takt_nr, 'Stacja_robocza': stacja,
                'Nr zlecenia prod/plan': 0, 'Numer sekwencyjny': 0, 'Nr kontraktu': 0,
                'Pozycja kontraktu': 0, 'Opis zlecenia': 0, 'Materiał': 0,
            })
        else:
            p = pojazd.iloc[0]
            wiersze_matrycy.append({
                'Takt': takt_nr, 'Stacja_robocza': stacja,
                'Nr zlecenia prod/plan': p['Nr zlec.prod/plan'], 'Numer sekwencyjny': p['Nr sekwencyjny'],
                'Nr kontraktu': p['Nr kontraktu'], 'Pozycja kontraktu': p['Poz. kontraktu'],
                'Opis zlecenia': p['Opis zlecenia'], 'Materiał': p['Materiał'],
            })
        pozycja += 1

df_matrix = pd.DataFrame(wiersze_matrycy)
df_matrix = df_matrix.astype({'Takt': int, 'Stacja_robocza': str, 'Nr zlecenia prod/plan': int,
                              'Numer sekwencyjny': int, 'Nr kontraktu': 'int64', 'Pozycja kontraktu': int})

# klucz do łączenia z planem SOP, np. '4500123-10'
df_matrix['kontrakt_pozycja'] = df_matrix['Nr kontraktu'].astype(str) + '-' + df_matrix['Pozycja kontraktu'].astype(str)

## 2. ZPP_SOP – ostatni pojazd w kontrakcie

Czasy montażu bierzemy z marszruty **ostatniego pojazdu** każdej pozycji kontraktu
(pojazd z najwyższym numerem sekwencyjnym).

In [ ]:
df_zpp_sop = pd.read_csv(sciezki['zpp_sop'], header=0, sep=';', low_memory=False)

# odrzucamy stare zlecenia
df_zpp_sop = df_zpp_sop.loc[df_zpp_sop['Nr zlecenia pojazdu'] > 100013000]

# najwyższy numer sekwencyjny w każdej pozycji kontraktu = ostatni pojazd
df_zpp_sop_zlecenia = df_zpp_sop.groupby(['Nr kontraktu', 'Pozycja kontraktu'], as_index=False).agg({
    'Numer sekwencyjny pojazdu': 'max',
})

# dołączenie numeru zlecenia tego pojazdu
df_zpp_sop_zlecenia = df_zpp_sop_zlecenia.merge(
    right=df_zpp_sop[['Nr kontraktu', 'Pozycja kontraktu', 'Numer sekwencyjny pojazdu', 'Nr zlecenia pojazdu']],
    on=['Nr kontraktu', 'Pozycja kontraktu', 'Numer sekwencyjny pojazdu'], how='left')

df_zpp_sop_zlecenia = df_zpp_sop_zlecenia.rename(columns={'Nr zlecenia pojazdu': 'Nr zlecenia ostatniego pojazdu'})
df_zpp_sop_zlecenia['kontrakt_pozycja'] = df_zpp_sop_zlecenia['Nr kontraktu'].astype(str) + '-' + df_zpp_sop_zlecenia['Pozycja kontraktu'].astype(str)

# jeden wiersz na pozycję kontraktu - gdyby numer sekwencyjny się powtórzył,
# wiersze matrycy zdublowałyby się przy łączeniu
df_zpp_sop_zlecenia = df_zpp_sop_zlecenia.drop_duplicates(subset='kontrakt_pozycja')

In [ ]:
# dodanie do matrycy numeru zlecenia ostatniego pojazdu w kontrakcie
df_matrix_sop = df_matrix.merge(right=df_zpp_sop_zlecenia[['kontrakt_pozycja', 'Nr zlecenia ostatniego pojazdu']],
                                on='kontrakt_pozycja', how='left')

# puste pola (stacja bez pojazdu) -> 0
df_matrix_sop['Nr zlecenia ostatniego pojazdu'] = df_matrix_sop['Nr zlecenia ostatniego pojazdu'].fillna(0).astype(int)

## 3. OPERACJE_ZP03 – potrzebne moce

Czas montażu z marszruty, przypisany do stacji przez słownik, dzielony przez czas taktu.

In [ ]:
df_coois = pd.read_csv(sciezki['operacje'], sep=';', encoding='utf-8')
df_coois = df_coois.loc[df_coois['Zlecenie produkcyjne'] > 100013000].copy()

# czas w formacie SAP ' 12,500' -> 12.5
df_coois['Czas montażu'] = tekst_na_liczbe(df_coois['Czas montażu'])

# suma czasu montażu dla każdego zlecenia i stanowiska SAP
df_coois_grouped = df_coois.groupby(['Zlecenie produkcyjne', 'Stanowisko robocze'], as_index=False).agg({
    'Czas montażu': 'sum'
})
df_coois_grouped = df_coois_grouped.rename(columns={'Zlecenie produkcyjne': 'Nr zlecenia ostatniego pojazdu'})

In [ ]:
# słownik: stanowisko SAP (STACJA_COOIS) -> stacja linii (STACJA_ZASOBY)
df_slownik_stacji = pd.read_excel(sciezki['slownik'], sheet_name='stacje', header=0)

df_coois_merged = df_coois_grouped.merge(df_slownik_stacji[['STACJA', 'STACJA_ZASOBY', 'STACJA_COOIS']],
                                         left_on='Stanowisko robocze', right_on='STACJA_COOIS', how='left')

# drugie grupowanie - kilka stanowisk SAP może należeć do jednej stacji linii
df_coois_grouped2 = df_coois_merged.groupby(['Nr zlecenia ostatniego pojazdu', 'STACJA_ZASOBY'], as_index=False).agg({
    'STACJA_COOIS': 'first',
    'STACJA': 'first',
    'Czas montażu': 'sum'
})

# potrzebne moce = czas montażu / czas taktu (liczba osób potrzebnych na stacji)
df_coois_grouped2['potrzebne_moce'] = round(df_coois_grouped2['Czas montażu'] / takt, 1)

In [ ]:
# połączenie matrycy z potrzebnymi mocami (po stacji i ostatnim pojeździe z kontraktu)
df_matrix_coois = df_matrix_sop.merge(
    right=df_coois_grouped2[['STACJA', 'Nr zlecenia ostatniego pojazdu', 'potrzebne_moce', 'STACJA_ZASOBY']],
    left_on=['Stacja_robocza', 'Nr zlecenia ostatniego pojazdu'],
    right_on=['STACJA_ZASOBY', 'Nr zlecenia ostatniego pojazdu'],
    how='left')

# kolumna STACJA_ZASOBY powtarza Stacja_robocza - usuwamy ją
df_matrix_coois = df_matrix_coois.drop(columns='STACJA_ZASOBY')

## 4. ZASOBY – obecni pracownicy

In [ ]:
# która brygada (A lub B) pracuje teraz
obecna_zmiana = ustal_obecna_zmiane(datetime.datetime.now())
print('Obecna zmiana:', obecna_zmiana)

In [ ]:
# lista pracowników z SAP
df_zasoby_sap = pd.read_csv(sciezki['zasoby'], sep=';', decimal=',', header=0)
df_zasoby_sap['Ilość godzin'] = tekst_na_liczbe(df_zasoby_sap['Ilość godzin']).fillna(0)

# słownik zmian: stanowisko + brygadzista (zmiana I/II) -> brygada A/B
df_nr_zmiany = pd.read_excel(sciezki['slownik'], sheet_name='nr_zmiany')
df_zasoby_workshift = pd.merge(df_nr_zmiany, df_zasoby_sap,
                               left_on=['brygadzista', 'stanowisko_robocze'],
                               right_on=['Numer zmiany (I lub II)', 'Stanowisko robocze'], how='left')

### Absencja na stanowiskach (arkusz `zasoby_total`)

Pomocnicza tabela do wizualizacji absencji. Uwzględnia wypożyczenia pracowników
między stanowiskami i szkolenia (szczegóły w `funkcje.uwzglednij_wypozyczenia`).

In [ ]:
df_zasoby_skorygowane = uwzglednij_wypozyczenia(df_zasoby_workshift)

# wszyscy pracownicy na stanowisku
df_zasoby_total = df_zasoby_skorygowane.groupby(['stanowisko_robocze', 'brygadzista'], as_index=False).agg({
    'nr_zmiany': 'first',
    'Imię i nazwisko pracownika': 'count'
}).rename(columns={'Imię i nazwisko pracownika': 'Total'})

# obecni pracownicy - liczeni z tej samej poprawionej listy, żeby nikt nie był liczony dwa razy
df_zasoby_present = df_zasoby_skorygowane[df_zasoby_skorygowane['Obecny'] == 'X']
df_zasoby_present = df_zasoby_present.groupby(['stanowisko_robocze', 'brygadzista'], as_index=False).agg({
    'nr_zmiany': 'first',
    'Imię i nazwisko pracownika': 'count'
}).rename(columns={'Imię i nazwisko pracownika': 'Obecni'})

df_zasoby_total = pd.merge(df_zasoby_total, df_zasoby_present, on=['stanowisko_robocze', 'brygadzista', 'nr_zmiany'], how='left')
df_zasoby_total['Obecni'] = df_zasoby_total['Obecni'].fillna(0)
df_zasoby_total['Absencja'] = (df_zasoby_total['Total'] - df_zasoby_total['Obecni']) / df_zasoby_total['Total']

# tylko bieżąca brygada i stanowiska hali C
df_zasoby_total = df_zasoby_total[df_zasoby_total['nr_zmiany'] == obecna_zmiana]
df_zasoby_total = df_zasoby_total[df_zasoby_total['stanowisko_robocze'].str.contains('C')]

df_zasoby_total = pd.merge(df_zasoby_total, df_slownik_stacji[['STACJA', 'STACJA_ZASOBY']].drop_duplicates(),
                           left_on='stanowisko_robocze', right_on='STACJA_ZASOBY', how='left')

zapisz_arkusz(df_zasoby_total, sciezki['raport'], 'zasoby_total')

### Bilans mocy (arkusz `matrix_coois_zasoby`)

`bilans_mocy = obecni − potrzebne_moce`. Wartość ujemna oznacza, że na stacji brakuje ludzi.

In [ ]:
# obecni pracownicy macierzyści (N, B, Z) na każdym stanowisku i zmianie
df_zasoby_grouped = df_zasoby_workshift[df_zasoby_workshift['Obecny'] == 'X']
df_zasoby_grouped = df_zasoby_grouped[df_zasoby_grouped['Typ pracownika'].isin(['N', 'B', 'Z'])].groupby(
    ['stanowisko_robocze', 'brygadzista'], as_index=False).agg({
    'nr_zmiany': 'first',
    'Obecny': 'count'
})

# dołączenie liczby obecnych z bieżącej brygady do matrycy
df_matrix_coois_zasoby = df_matrix_coois.merge(
    right=df_zasoby_grouped[df_zasoby_grouped['nr_zmiany'] == obecna_zmiana][['stanowisko_robocze', 'nr_zmiany', 'Obecny']],
    left_on='Stacja_robocza', right_on='stanowisko_robocze', how='left')
df_matrix_coois_zasoby = df_matrix_coois_zasoby.drop(columns='stanowisko_robocze')

# stacje bez danych -> 0
df_matrix_coois_zasoby['nr_zmiany'] = df_matrix_coois_zasoby['nr_zmiany'].fillna(obecna_zmiana)
df_matrix_coois_zasoby['Obecny'] = df_matrix_coois_zasoby['Obecny'].fillna(0)
df_matrix_coois_zasoby['potrzebne_moce'] = df_matrix_coois_zasoby['potrzebne_moce'].fillna(0)

df_matrix_coois_zasoby['bilans_mocy'] = df_matrix_coois_zasoby['Obecny'] - df_matrix_coois_zasoby['potrzebne_moce']

zapisz_arkusz(df_matrix_coois_zasoby, sciezki['raport'], 'matrix_coois_zasoby')
df_matrix_coois_zasoby.head()

## 5. Zapis do bazy SQLite

Oprócz pliku Excel dla Power BI oba wyniki trafiają do bazy SQLite (jeden plik `.db`).
Z bazy można korzystać zapytaniami SQL, np.:

```sql
SELECT Stacja_robocza, MIN(bilans_mocy) AS najwiekszy_brak
FROM matrix_coois_zasoby
GROUP BY Stacja_robocza
ORDER BY najwiekszy_brak;
```

In [ ]:
zapisz_do_bazy(df_matrix_coois_zasoby, sciezki['baza'], 'matrix_coois_zasoby')
zapisz_do_bazy(df_zasoby_total, sciezki['baza'], 'zasoby_total')
print('Zapisano wyniki do bazy:', sciezki['baza'])